Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnx
import sys
import os
os.chdir("../../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Load Data

In [3]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time']) 
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df) 

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [4]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

In [5]:
# Create test value
manualX = X_test
manualY = y_test
print(manualY)
print(manualX)

[-0.09924674 -0.09546565 -0.12112098 ... -0.01743513 -0.03103252
 -0.04362074]
[[-0.09924674  0.48484848  0.93       ...  0.          0.
   1.        ]
 [-0.09924674  0.48484848  0.93       ...  0.          0.25881905
   0.96592583]
 [-0.09546565  0.48484848  0.93       ...  0.          0.5
   0.8660254 ]
 ...
 [-0.00225057  0.57373737  0.894      ...  1.         -0.70710678
   0.70710678]
 [-0.01743513  0.57373737  0.894      ...  1.         -0.5
   0.8660254 ]
 [-0.03103252  0.57373737  0.894      ...  1.         -0.25881905
   0.96592583]]


MLP Model

In [6]:
import onnx
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
graph = onnx_model.graph
#print(onnx_model)
weights = {}
for initializer in onnx_model.graph.initializer:
    weights[initializer.name] = onnx.numpy_helper.to_array(initializer)



In [7]:
#print(graph)
#print(onnx_model)
#print(weights)

In [8]:
import onnx
import onnxruntime as ort
import numpy as np
from sklearn.neural_network import MLPRegressor



# Hidden layer parameters
hidden_weights = weights["coefficient"]
hidden_bias = weights["intercepts"]

# Output layer parameters
output_weights = weights["coefficient1"]
output_bias = weights["intercepts1"]


# The second dimension of the weight matrix is the number of neurons
hidden_layer_size = hidden_weights.shape[1]


In [9]:
# Instantiate model
mlp_model = MLPRegressor(
    hidden_layer_sizes=(hidden_layer_size,),
    activation='relu', 
    solver='adam',     
    max_iter=1,       
    random_state=42    
)

dummy_input = np.zeros((1, 13), dtype=np.float32)
dummy_output = np.zeros((1,), dtype=np.float32)
mlp_model.partial_fit(dummy_input, dummy_output)

# Assign known weights and biases

mlp_model.coefs_[0] = hidden_weights
print(mlp_model.coefs_[0].shape)
mlp_model.intercepts_[0] = hidden_bias
print(mlp_model.intercepts_[0].shape)
mlp_model.coefs_[1] = output_weights
print(mlp_model.coefs_[1].shape)
mlp_model.intercepts_[1] = output_bias
print(mlp_model.intercepts_[1].shape)


(13, 500)
(1, 500)
(500, 1)
(1, 1)


In [19]:
data = manualX 
print(data)
mlp_model.predict(data) 
pred_out = mlp_model.predict(data) 
print(pred_out)

[[-0.09924674  0.48484848  0.93       ...  0.          0.
   1.        ]
 [-0.09924674  0.48484848  0.93       ...  0.          0.25881905
   0.96592583]
 [-0.09546565  0.48484848  0.93       ...  0.          0.5
   0.8660254 ]
 ...
 [-0.00225057  0.57373737  0.894      ...  1.         -0.70710678
   0.70710678]
 [-0.01743513  0.57373737  0.894      ...  1.         -0.5
   0.8660254 ]
 [-0.03103252  0.57373737  0.894      ...  1.         -0.25881905
   0.96592583]]
[-0.1134601  -0.11022906 -0.11441346 ... -0.01580448 -0.03803045
 -0.0439702 ]


In [15]:
inputsL = manualX
#inputsL.append(manualX.tolist())
#print(inputsL)

import onnxruntime as rt
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
model_inv = onnx_model
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(inputsL, dtype=np.float32) 
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
print(outputs)

[array([[-0.1134601 ],
       [-0.11022907],
       [-0.11441348],
       ...,
       [-0.01580443],
       [-0.03803042],
       [-0.04397026]], shape=(70128, 1), dtype=float32)]


In [25]:
from sklearn.metrics import mean_absolute_error
old_out = []
for item in outputs[0]:
    old_out.append(item[0])
print(mean_absolute_error(old_out,pred_out))

2.619273435072886e-08
